# Iris Dataset — Complete ML Pipeline
### Multi-class Classification using Logistic Regression
---
**Dataset:** Iris — 150 flowers, 3 species, 4 features  
**Goal:** Predict which species a flower belongs to  
**Algorithm:** Logistic Regression  
> Built by Ather Assadullah — github.com/ather-ops

## What is the Iris Dataset?

The Iris dataset has **150 rows** and **4 features**. Each row is one flower.

| Feature | What it means | Example value |
|---------|--------------|---------------|
| sepal length (cm) | Length of the outer petal | 5.1 cm |
| sepal width (cm) | Width of the outer petal | 3.5 cm |
| petal length (cm) | Length of the inner petal | 1.4 cm |
| petal width (cm) | Width of the inner petal | 0.2 cm |

**Target (Y) — 3 classes:**
- `0` = Setosa
- `1` = Versicolor  
- `2` = Virginica

**Our job:** Look at 4 measurements → predict which of the 3 species it is.  
This is called **Multi-class Classification**.

## Step 1 — Import Libraries

**Bug fixed here:** Your original code had `iris.feature_target` — this does NOT exist.  
The correct attribute is `iris.target_names`.

In [ ]:
# Step 1: Import all important libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, classification_report,
                              confusion_matrix, ConfusionMatrixDisplay)

print("All libraries imported successfully")
print("="*60)

## Step 2 — Load the Iris Dataset

`load_iris()` returns a special object called a **Bunch**.  
Think of it like a dictionary with keys:
- `iris.data` → the 4 features (X)
- `iris.target` → the labels (Y) — 0, 1, or 2
- `iris.feature_names` → names of the 4 columns
- `iris.target_names` → names of the 3 species  ← **NOT feature_target** (that was your bug)

In [ ]:
# Step 2: Load data
iris = load_iris()

# Step 3: Features and target
X = iris.data     # shape: (150, 4) — 150 flowers, 4 measurements each
Y = iris.target   # shape: (150,)   — 150 labels: 0, 1, or 2

feature_names = iris.feature_names   # ['sepal length', 'sepal width', 'petal length', 'petal width']
target_names  = iris.target_names    # ['setosa', 'versicolor', 'virginica']
                                     # BUG FIX: was iris.feature_target — does not exist!

print("Feature names:", feature_names)
print("="*60)
print("Target names:", target_names)
print("="*60)
print("Type of X:", type(X), "| Shape:", X.shape)
print("Type of Y:", type(Y), "| Shape:", Y.shape)
print("="*60)
print("First 5 rows of X (features):")
print(X[:5])
print("="*60)
print("First 5 values of Y (labels):", Y[:5])
print("Each number means: 0=Setosa, 1=Versicolor, 2=Virginica")

## Step 3 — Exploratory Data Analysis (EDA)

Before training, always **look at the data**.  
We need to understand:
- How many samples per class?
- Are features on similar scales?
- Any obvious patterns?

In [ ]:
# EDA — understand data before training

# Convert to DataFrame for easy viewing
df = pd.DataFrame(X, columns=feature_names)
df['species'] = Y
df['species_name'] = df['species'].map({0:'Setosa', 1:'Versicolor', 2:'Virginica'})

print("Dataset shape:", df.shape)
print("="*60)
print("First 5 rows:")
print(df.head())
print("="*60)
print("Class distribution (how many of each species):")
print(df['species_name'].value_counts())
print("="*60)
print("Basic statistics:")
print(df.describe().round(2))

### EDA Visualization — 4 Panel Plot

**What each plot shows:**
- **Top left:** Petal length for each species — are they separable?
- **Top right:** Petal width for each species
- **Bottom left:** Sepal length comparison
- **Bottom right:** Scatter — petal length vs petal width  

If classes are visually separate → model will do well.

In [ ]:
# EDA Visualization — 4 panel
fig, axes = plt.subplots(2, 2, figsize=(12, 8))
fig.patch.set_facecolor('#0a0a14')
colors = ['#f97316', '#06b6d4', '#a855f7']
species_names = ['Setosa', 'Versicolor', 'Virginica']

for ax in axes.flat:
    ax.set_facecolor('#0f172a')
    ax.tick_params(colors='#94a3b8')
    ax.xaxis.label.set_color('#94a3b8')
    ax.yaxis.label.set_color('#94a3b8')
    for spine in ax.spines.values():
        spine.set_color('#1e293b')

# Plot 1: Petal length per species
for i, (name, color) in enumerate(zip(species_names, colors)):
    axes[0,0].hist(X[Y==i, 2], bins=15, alpha=0.7, color=color, label=name)
axes[0,0].set_title('Petal Length per Species', color='white')
axes[0,0].set_xlabel('Petal Length (cm)')
axes[0,0].legend(facecolor='#1e293b', labelcolor='white')

# Plot 2: Petal width per species
for i, (name, color) in enumerate(zip(species_names, colors)):
    axes[0,1].hist(X[Y==i, 3], bins=15, alpha=0.7, color=color, label=name)
axes[0,1].set_title('Petal Width per Species', color='white')
axes[0,1].set_xlabel('Petal Width (cm)')
axes[0,1].legend(facecolor='#1e293b', labelcolor='white')

# Plot 3: Sepal length per species
for i, (name, color) in enumerate(zip(species_names, colors)):
    axes[1,0].hist(X[Y==i, 0], bins=15, alpha=0.7, color=color, label=name)
axes[1,0].set_title('Sepal Length per Species', color='white')
axes[1,0].set_xlabel('Sepal Length (cm)')
axes[1,0].legend(facecolor='#1e293b', labelcolor='white')

# Plot 4: Scatter — petal length vs petal width
for i, (name, color) in enumerate(zip(species_names, colors)):
    axes[1,1].scatter(X[Y==i, 2], X[Y==i, 3], color=color, label=name, alpha=0.8, s=40)
axes[1,1].set_title('Petal Length vs Petal Width', color='white')
axes[1,1].set_xlabel('Petal Length (cm)')
axes[1,1].set_ylabel('Petal Width (cm)')
axes[1,1].legend(facecolor='#1e293b', labelcolor='white')

plt.suptitle('Iris Dataset — EDA', color='white', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('iris_eda.png', dpi=120, bbox_inches='tight', facecolor='#0a0a14')
plt.show()
print("Observation: Setosa is clearly separate. Versicolor and Virginica overlap a bit.")

## Step 4 — Train Test Split

We split data into two parts:
- **Training set:** Model learns from this
- **Test set:** We evaluate the model on this — model has NEVER seen it

**Your original code used `test_size=0.4`** — that means 40% test, 60% train.  
Standard is 20-30% test. We use 0.2 here (better practice).

**`stratify=Y`** — makes sure each class has equal representation in both splits.  
This is important for Iris because all 3 classes should be in train and test.

In [ ]:
# Step 4: Train Test Split
X_train, X_test, Y_train, Y_test = train_test_split(
    X, Y,
    test_size=0.2,      # 20% test, 80% train (better than 0.4)
    random_state=42,    # reproducible results
    stratify=Y          # equal class distribution in both splits
)

print("X_train shape:", X_train.shape, "← 80% of 150 = 120 samples")
print("X_test shape: ", X_test.shape,  "← 20% of 150 = 30 samples")
print("Y_train shape:", Y_train.shape)
print("Y_test shape: ", Y_test.shape)
print("="*60)

# Check class distribution in both splits
print("Class distribution in Y_train:", np.bincount(Y_train))
print("Class distribution in Y_test: ", np.bincount(Y_test))
print("Each class has equal samples — stratify=Y worked correctly")

## Step 5 — Feature Scaling (StandardScaler)

**Why scale?**  
Sepal length is ~5-8 cm. Petal width is ~0.1-2.5 cm.  
Without scaling, the model thinks sepal length is more important just because its values are bigger.  
StandardScaler brings everything to **mean=0, std=1**.

**CRITICAL RULE:**
- `fit_transform` on **train only** ← learns mean and std from training data
- `transform` on **test only** ← applies same mean and std from training data
- NEVER fit on test data — that would be data leakage!

In [ ]:
# Step 5: Feature Scaling
scaler = StandardScaler()

# fit_transform on TRAIN — learns mean and std from training data
X_train_scaled = scaler.fit_transform(X_train)

# transform on TEST — applies training mean/std to test (NO leakage)
X_test_scaled = scaler.transform(X_test)

print("Before scaling — X_train first row:", X_train[0].round(2))
print("After scaling  — X_train first row:", X_train_scaled[0].round(4))
print("="*60)
print("Mean of scaled training data (should be ~0):", X_train_scaled.mean(axis=0).round(4))
print("Std  of scaled training data (should be ~1):", X_train_scaled.std(axis=0).round(4))

## Step 6 — Model Training (Logistic Regression)

**Why Logistic Regression for Iris?**
- 3 classes → Logistic Regression uses **One-vs-Rest (OvR)** strategy internally
- It trains 3 separate classifiers: Is it Setosa? Is it Versicolor? Is it Virginica?
- Then picks the class with highest probability

`max_iter=200` — gives model enough iterations to converge.

In [ ]:
# Step 6: Model Training
model = LogisticRegression(max_iter=200, random_state=42)
model.fit(X_train_scaled, Y_train)

print("Model trained successfully")
print("="*60)
print("Model classes:", model.classes_, "→ [0=Setosa, 1=Versicolor, 2=Virginica]")
print("Number of iterations taken:", model.n_iter_)

## Step 7 — Predictions

Two types of predictions:
- `model.predict()` → gives the final class (0, 1, or 2)
- `model.predict_proba()` → gives probability for EACH class  
  Example: [0.02, 0.15, 0.83] means → 2% Setosa, 15% Versicolor, 83% Virginica → predicts Virginica

In [ ]:
# Step 7: Predictions
Y_pred = model.predict(X_test_scaled)
Y_prob = model.predict_proba(X_test_scaled)

print("First 10 actual labels:   ", Y_test[:10])
print("First 10 predicted labels:", Y_pred[:10])
print("="*60)
print("Match?", Y_test[:10] == Y_pred[:10])
print("="*60)
print("Probabilities for first 3 samples:")
print("         Setosa  Versicolor  Virginica")
for i in range(3):
    prob = Y_prob[i]
    print(f"Sample {i+1}: {prob[0]:.3f}    {prob[1]:.3f}       {prob[2]:.3f}  → Predicts: {target_names[Y_pred[i]]}")

## Step 8 — Model Evaluation

**Accuracy** = what % of predictions were correct overall

**Classification Report** shows for each class:
- **Precision** = Of all flowers I predicted as Setosa, how many were actually Setosa?
- **Recall** = Of all actual Setosa flowers, how many did I correctly identify?
- **F1-score** = Balance between Precision and Recall

**Confusion Matrix** = table showing actual vs predicted
- Diagonal = correct predictions
- Off-diagonal = mistakes

In [ ]:
# Step 8: Evaluation
accuracy = accuracy_score(Y_test, Y_pred)
print(f"Accuracy: {accuracy:.4f} ({accuracy*100:.2f}%)")
print("="*60)
print("Classification Report:")
print(classification_report(Y_test, Y_pred,
                             target_names=['Setosa','Versicolor','Virginica']))
print("="*60)
print("Confusion Matrix (rows=actual, cols=predicted):")
cm = confusion_matrix(Y_test, Y_pred)
print(cm)

### Confusion Matrix — Visual

**How to read:**
- Row = actual class
- Column = predicted class
- Diagonal cells (top-left to bottom-right) = CORRECT predictions
- Any number off the diagonal = WRONG prediction

Perfect model = only diagonal has numbers, everything else is 0.

In [ ]:
# Confusion Matrix — Visual
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.patch.set_facecolor('#0a0a14')

# Plot 1: Confusion matrix heatmap
cm = confusion_matrix(Y_test, Y_pred)
ax1 = axes[0]
ax1.set_facecolor('#0f172a')
im = ax1.imshow(cm, cmap='Oranges')
ax1.set_xticks([0,1,2])
ax1.set_yticks([0,1,2])
ax1.set_xticklabels(['Setosa','Versicolor','Virginica'], color='white', fontsize=9)
ax1.set_yticklabels(['Setosa','Versicolor','Virginica'], color='white', fontsize=9)
ax1.set_xlabel('Predicted', color='#f97316', fontsize=11)
ax1.set_ylabel('Actual', color='#f97316', fontsize=11)
ax1.set_title('Confusion Matrix', color='white', fontsize=12, fontweight='bold')
for i in range(3):
    for j in range(3):
        color = 'black' if cm[i,j] > cm.max()/2 else 'white'
        ax1.text(j, i, str(cm[i,j]), ha='center', va='center',
                 color=color, fontsize=14, fontweight='bold')

# Plot 2: Probability distribution for each class
ax2 = axes[1]
ax2.set_facecolor('#0f172a')
colors_prob = ['#f97316', '#06b6d4', '#a855f7']
for i, (name, color) in enumerate(zip(['Setosa','Versicolor','Virginica'], colors_prob)):
    ax2.scatter(range(len(Y_test)), Y_prob[:,i], alpha=0.6, color=color,
                label=name, s=25)
ax2.set_title('Prediction Probabilities per Class', color='white', fontsize=12)
ax2.set_xlabel('Test Sample Index', color='#94a3b8')
ax2.set_ylabel('Probability', color='#94a3b8')
ax2.legend(facecolor='#1e293b', labelcolor='white')
ax2.tick_params(colors='#94a3b8')
ax2.set_facecolor('#0f172a')
for spine in ax2.spines.values():
    spine.set_color('#1e293b')

plt.suptitle('Iris Classification — Results', color='white', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('iris_results.png', dpi=120, bbox_inches='tight', facecolor='#0a0a14')
plt.show()

## Step 9 — Predict a New Flower

This is the most important step for production use.  
We have a new flower with these measurements:
- Sepal length: 5.1 cm
- Sepal width: 3.5 cm
- Petal length: 1.4 cm
- Petal width: 0.2 cm

What species is it?

In [ ]:
# Step 9: Predict a new flower
import warnings
warnings.filterwarnings('ignore')

# New flower measurements
new_flower = np.array([[5.1, 3.5, 1.4, 0.2]])  # shape must be (1, 4)

# IMPORTANT: scale new data using the SAME scaler (already fitted)
new_flower_scaled = scaler.transform(new_flower)

# Predict
prediction = model.predict(new_flower_scaled)
probabilities = model.predict_proba(new_flower_scaled)

print("New flower measurements:", new_flower[0])
print("="*60)
print(f"Predicted species: {target_names[prediction[0]]} (class {prediction[0]})")
print("="*60)
print("Probabilities:")
for name, prob in zip(target_names, probabilities[0]):
    bar = '#' * int(prob * 30)
    print(f"  {name:12}: {prob:.4f} ({prob*100:.1f}%) |{bar}|")

## Summary — What We Did

| Step | Action | Why |
|------|--------|-----|
| 1 | Import libraries | Get all tools ready |
| 2 | Load Iris dataset | 150 flowers, 4 features, 3 classes |
| 3 | EDA | Understand data before training |
| 4 | Train-test split | 80% train, 20% test, stratified |
| 5 | StandardScaler | Balance feature scales — fit on train only |
| 6 | LogisticRegression | Multi-class classification |
| 7 | predict + predict_proba | Class label + probability for each class |
| 8 | Evaluate | Accuracy, classification report, confusion matrix |
| 9 | New prediction | Real-world usage |

---

## Bug Fixes from Your Original Code

| Bug | Original | Fixed |
|-----|----------|-------|
| Wrong attribute | `iris.feature_target` | `iris.target_names` |
| High test size | `test_size=0.4` | `test_size=0.2` |
| No stratify | `train_test_split(X,Y,...)` | Added `stratify=Y` |
| No max_iter | `LogisticRegression()` | `LogisticRegression(max_iter=200)` |
| No visualization | Only print | Full EDA + confusion matrix plots |

---
**Built by Ather Assadullah — github.com/ather-ops**